In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from datetime import datetime, timezone
import json, os, pathlib, subprocess, sys
REPO_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
BRANCH = 'Content-V10'
EXPECTED_EXACT = '15f8d743b7a88e18eaf9e0826f33bbd1bc23231f'
RUNNER_MODULE = 'experiments.run_content_v10_calibration'
SOURCE = pathlib.Path('/content/cegwm-content-v10-calibration-source')
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/CEG-WM/Content')
CAPTURE_LIMIT = 4096
SUMMARY_PREFIX = 'CEGWM_CONTENT_V10_CALIBRATION_SUMMARY '
RUN_UTC = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
LOCAL = pathlib.Path('/content') / f'Content-V10-Calibration-{EXPECTED_EXACT[:7]}-{RUN_UTC}-local'
DRIVE_TARGET = DRIVE_ROOT / f'Content-V10-Calibration-{EXPECTED_EXACT[:7]}-{RUN_UTC}'


In [ ]:
if SOURCE.exists() or LOCAL.exists() or DRIVE_TARGET.exists(): raise FileExistsError('create-only path')
subprocess.run(['git','clone','--no-single-branch','--branch',BRANCH,REPO_URL,str(SOURCE)],check=True)
def git(*args): return subprocess.run(['git',*args],cwd=SOURCE,check=True,capture_output=True,text=True).stdout.strip()
if git('branch','--show-current') != BRANCH or git('rev-parse','HEAD') != EXPECTED_EXACT or git('status','--porcelain'): raise RuntimeError('checkout identity')
subprocess.run([sys.executable,'-m','pip','install',str(SOURCE)],check=True)
if git('branch','--show-current') != BRANCH or git('rev-parse','HEAD') != EXPECTED_EXACT or git('status','--porcelain') or LOCAL.exists() or DRIVE_TARGET.exists(): raise RuntimeError('post-install identity')
from google.colab import userdata
child_env={k:v for k,v in os.environ.items() if not any(x in k.upper() for x in ('TOKEN','KEY','SECRET','PASSWORD','CREDENTIAL'))}; root_key=token=''
try:
    root_key=userdata.get('CEG_WM_ROOT_KEY'); token=userdata.get('HF_TOKEN'); child_env['CEG_WM_ROOT_KEY']=root_key; child_env['HF_TOKEN']=token
    p=subprocess.Popen([sys.executable,'-m',RUNNER_MODULE,'--repo-root',str(SOURCE),'--expected-exact',EXPECTED_EXACT,'--local-work-root',str(LOCAL),'--artifact-sink',str(DRIVE_TARGET)],cwd=SOURCE,env=child_env,stdout=subprocess.PIPE,stderr=subprocess.DEVNULL)
finally:
    child_env.pop('CEG_WM_ROOT_KEY',None); child_env.pop('HF_TOKEN',None); root_key=token=''
summary=None; summary_count=0
for raw_line in iter(p.stdout.readline,b''):
    if len(raw_line)>CAPTURE_LIMIT: raise RuntimeError('runner line bound')
    line=raw_line.decode('utf-8','strict').strip()
    if line.startswith(SUMMARY_PREFIX): summary=json.loads(line[len(SUMMARY_PREFIX):]); summary_count+=1
rc=p.wait()
if summary_count!=1 or not isinstance(summary,dict): raise RuntimeError('terminal summary contract')


In [ ]:
required=('status','completeness','scientific_status','claim_ceiling','exact','manifest_digest','fixed_units','committed_units','failed_units','pair_count','asset_path','sidecar_path','asset_sha256')
if set(summary) != set(required): raise RuntimeError('terminal summary fields')
if summary['claim_ceiling'] != 'v10_calibration_asset_generation_only_no_efficacy_claim' or summary['exact'] != EXPECTED_EXACT: raise RuntimeError('terminal identity')
if rc==0 and summary['status']=='complete' and summary['completeness']=='complete' and summary['scientific_status']=='not_adjudicated' and (summary['fixed_units'],summary['committed_units'],summary['failed_units'],summary['pair_count'])==(32,32,0,1056) and all(isinstance(summary[k],str) and summary[k] for k in ('asset_path','sidecar_path','asset_sha256')) and pathlib.Path(summary['asset_path']).is_file() and pathlib.Path(summary['sidecar_path']).is_file():
    print('CEGWM_CONTENT_V10_CALIBRATION_ARTIFACT '+json.dumps({'execution_exact':EXPECTED_EXACT,'asset_path':summary['asset_path'],'sidecar_path':summary['sidecar_path'],'asset_sha256':summary['asset_sha256'],'completeness':'complete','scientific_status':'not_adjudicated'},sort_keys=True,separators=(',',':')))
elif rc==2 and summary['status']=='incomplete' and summary['completeness']=='incomplete' and summary['scientific_status']=='not_evaluable' and (summary['committed_units']<32 or summary['failed_units']>0) and summary['pair_count']<1056 and summary['asset_path'] is None and summary['sidecar_path'] is None and summary['asset_sha256'] is None:
    print('CEGWM_CONTENT_V10_CALIBRATION_INCOMPLETE '+json.dumps({'execution_exact':EXPECTED_EXACT,'completeness':'incomplete','scientific_status':'not_evaluable'},sort_keys=True,separators=(',',':')))
else: raise RuntimeError('runner completion contract')
